In [ ]:
"""
Notebook 02: Segmentation, Elasticity Estimation, and CLV
==========================================================
Estimates price elasticity by product bucket and category using
log-log OLS regression with fixed effects. Models customer propensity
to repeat purchase using logistic regression. Calculates customer
lifetime value by segment.

Paper: Profit-Aware Pricing in Two-Sided Marketplaces
Author: Dinesh R. Poddaturi, Ph.D.
SSRN: https://ssrn.com/abstract=6502262

Requires: Run notebook 01 first to generate processed data files.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported and display options set.")

In [ ]:
# Loading  the saved data from data exploration

# Load the cleaned and preprocessed data
print("Loading the main analysis Datasets")
orders_full = pd.read_pickle('../data/processed/orders_full_clean.pkl')
print(f"orders_full: {len(orders_full)} rows, {orders_full.shape[1]} columns")

orders_full_extended = pd.read_pickle('../data/processed/orders_full_extended.pkl')
print(f"orders_full_extended: {len(orders_full_extended)} rows, {orders_full_extended.shape[1]} columns")

order_items_with_bucket = pd.read_pickle('../data/processed/order_items_with_bucket.pkl')
print(f"order_items_with_bucket: {len(order_items_with_bucket)} rows, {order_items_with_bucket.shape[1]} columns")

# Quantity datasets
print("Quantity/Demand datasets")
product_price_quantity = pd.read_pickle('../data/processed/product_price_quantity.pkl')
print(f"product_price_quantity: {len(product_price_quantity)} rows, {product_price_quantity.shape[1]} columns")

product_quantity_monthly = pd.read_pickle('../data/processed/product_quantity_monthly.pkl')
print(f"product_quantity_monthly: {len(product_quantity_monthly)} rows, {product_quantity_monthly.shape[1]} columns")

bucket_demand_monthly = pd.read_pickle('../data/processed/bucket_demand_monthly.pkl')
print(f"bucket_demand_monthly: {len(bucket_demand_monthly)} rows, {bucket_demand_monthly.shape[1]} columns")

# Repeat analysis
print("Repeat purchase datasets")
repeat_purchases = pd.read_pickle('../data/processed/repeat_purchases.pkl')
print(f"repeat_purchases: {len(repeat_purchases)} rows, {repeat_purchases.shape[1]} columns")

repeat_customer_orders = pd.read_pickle('../data/processed/repeat_customer_orders.pkl')
print(f"repeat_customer_orders: {len(repeat_customer_orders)} rows, {repeat_customer_orders.shape[1]} columns")

customer_bucket_diversity = pd.read_pickle('../data/processed/customer_bucket_diversity.pkl')
print(f"customer_bucket_diversity: {len(customer_bucket_diversity)} rows, {customer_bucket_diversity.shape[1]} columns")

In [ ]:
# ============================================
# QUICK DATA VALIDATION
# ============================================

print("\n Data Validation & Summary")
print("="*60)

# Date range
date_min = orders_full_extended['order_purchase_timestamp'].min()
date_max = orders_full_extended['order_purchase_timestamp'].max()
print(f"\n  Period: {date_min.date()} to {date_max.date()}")

# Counts
print(f"\nCounts:")
print(f"Orders: {orders_full_extended['order_id'].nunique():,}")
print(f"Customers: {orders_full_extended['customer_unique_id'].nunique():,}")
print(f"Products: {orders_full_extended['product_id'].nunique():,}")
print(f"Sellers: {orders_full_extended['seller_id'].nunique():,}")

# Buckets
print(f"\nBuckets:")
for bucket, count in orders_full_extended['bucket'].value_counts().head(5).items():
    pct = count / len(orders_full_extended) * 100
    print(f"{bucket:<25} {count:>7,} ({pct:>5.1f}%)")

# Repeat rate
repeat_rate = (repeat_purchases['num_orders'] >= 2).mean() * 100
print(f"\nRepeat Rate: {repeat_rate:.2f}%")

In [ ]:
# Nested logit feasibility check

# Bucket Samples Sizes
print("Bucket sample sizes: need 1000+ records for reliable nested logit estimation")
bucket_sizes = orders_full_extended.groupby('bucket').size().sort_values(ascending=False)
for bucket, count in bucket_sizes.items():
    status = "OK" if count >= 1000 else "TOO SMALL"
    print(f"  {status} {bucket:<25} {count:>7,} transactions")

sufficient = (bucket_sizes >= 1000).sum()
print(f"\n Result: {sufficient}/{len(bucket_sizes)} buckets have sufficient data for nested logit estimation.")


# Price variation within buckets
print("\n price variation within buckets: need CV>0.1")
print("-" * 70)
print(f"{'Bucket':<25} {'Mean Price':>12} {'Std Dev':>12} {'CV':>8} {'Status'}")
print("-" * 70)

good_variation_count = 0
for bucket in bucket_sizes.index:
    bucket_data = orders_full_extended[orders_full_extended['bucket'] == bucket]
    mean_price = bucket_data['price'].mean()
    std_price = bucket_data['price'].std()
    cv = std_price / mean_price if mean_price > 0 else 0
    status = "OK" if cv >= 0.1 else "TOO LOW"
    if cv > 0.1:
        good_variation_count += 1
    print(f"{bucket:<25} {mean_price:>12.2f} {std_price:>12.2f} {cv:>8.2f} {status}")

print(f"\n Result: {good_variation_count}/{len(bucket_sizes)} buckets have sufficient price variation for nested logit estimation.")

# Product diversity within buckets
print("\n product diversity within buckets: need >50 unique products")

diverse_count = 0
for bucket in bucket_sizes.index:
    bucket_data = orders_full_extended[orders_full_extended['bucket'] == bucket]
    unique_products = bucket_data['product_id'].nunique()
    status = "DIVERSE" if unique_products > 100 else "OK" if unique_products> 50 else "TOO LOW"
    if unique_products > 50:
        diverse_count += 1
    print(f"{bucket:<25} {unique_products:>12} unique products {status}")

print(f"\n Result: {diverse_count}/{len(bucket_sizes)} buckets have 50+ products.")

# Cross-bucket shopping behavior
print("\n cross-bucket shopping behavior: this tests the substitution patterns needed for nested logit estimation")

multi_bucket_customers = customer_bucket_diversity[customer_bucket_diversity['num_buckets'] > 1]

if len(multi_bucket_customers) > 0:
    total_repeat = len(customer_bucket_diversity)
    multi_bucket_pct = len(multi_bucket_customers) / total_repeat * 100

    print(f"Repeat Customers: {total_repeat:,}")
    print(f"Shopping 1 bucket only: {total_repeat - len(multi_bucket_customers):,} ({100 - multi_bucket_pct:.1f}%)")
    print(f"Shopping multiple buckets: {len(multi_bucket_customers):,} ({multi_bucket_pct:.1f}%)")

    avg_buckets = customer_bucket_diversity['num_buckets'].mean()
    print(f"Average buckets per repeat customer: {avg_buckets:.2f}")

    if multi_bucket_pct > 20:
        print("Result: Sufficient cross-bucket shopping behavior for nested logit estimation.")
    else:
        print("Result: Limited cross-bucket shopping behavior may challenge nested logit estimation.")
else:
    print("No customers shop across multiple buckets, which may severely limit nested logit estimation.")


# Final Recommendation

viable_buckets = bucket_sizes[bucket_sizes >= 1000].index.tolist()

print(f"Buckets viable for nested logit: {len(viable_buckets)}")
print(f"{viable_buckets[:5]}")

if len(viable_buckets) >= 5:
    print("Nested logit estimation is feasible with these buckets.")
elif len(viable_buckets) > 3:
    print("Nested logit estimation may be possible but with limited buckets.")
else:
    print("Nested logit estimation may not be feasible due to insufficient data in buckets. Skip nested logit and use simple log-log demand estimation instead.")

In [ ]:
# Elasticity Data quality check

# Quantity distribution
print("\n Quantity distribution check: looking for reasonable mean and variance")
print(f"\n Total product-price combinations: {len(product_price_quantity):,}")
print("\n Quantity statistics")
print(product_price_quantity['quantity'].describe())

# Visualize quantity distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(product_price_quantity['quantity'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Quantity sold per Product-Price Combination')
axes[0].set_xlabel('Quantity Sold')
axes[0].set_ylabel('Frequency')
axes[0].axvline(product_price_quantity['quantity'].median(), color='red', 
                linestyle='dashed', linewidth=1, label=f'Median: {product_price_quantity["quantity"].median():.2f}')
axes[0].legend()

# Log scale
axes[1].hist(np.log1p(product_price_quantity['quantity']), bins=50, color='salmon', edgecolor='black', alpha=0.7)
axes[1].set_title('Log Distribution (for log-log regression)')
axes[1].set_xlabel('Log(Quantity Sold + 1)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Products with sufficient observations for elasticity estimation
min_qyt_threshold = 10
sufficient_quantity = product_price_quantity[product_price_quantity['quantity'] >= min_qyt_threshold]
print(f"Products with quantity >= {min_qyt_threshold}: {len(sufficient_quantity):,} ({len(sufficient_quantity)/len(product_price_quantity)*100:.1f}%)")


# Price variation by bucket

print("\n Price variation by bucket: looking for sufficient variation within buckets for elasticity estimation")
print(f"{'Bucket': <25} {'Products':<12} {'Avg Qty : <12'} {'Price Range'}")

for bucket in ['LEISURE_LIFESTYLE', 'HOME_ESSENTIALS', 'PERSONAL_CARE', 'ELECTRONICS_TECH', 'AUTO_TOOLS']:
    bucket_data = product_price_quantity[product_price_quantity['bucket'] == bucket]
    if len(bucket_data) > 0:
        n_products = bucket_data['product_id'].nunique()
        avg_qty = bucket_data['quantity'].mean()
        price_min = bucket_data['price'].min()
        price_max = bucket_data['price'].max()
        
        print(f"{bucket:<25} {n_products:>6,}      {avg_qty:>8.1f}      {price_min:>6.2f} - {price_max:>8.2f}")


# Same product price variation check

print("\n Same product price variation check: looking for products sold at multiple price points for elasticity estimation")

# Count how many products have multiple price points
product_price_counts = product_price_quantity.groupby('product_id').size()
products_multiple_prices = (product_price_counts > 1).sum()
total_products = len(product_price_counts)

print(f"Products with multiple price points: {products_multiple_prices:,} ({products_multiple_prices/total_products*100:.1f}%)")
print(f"Products with only one price point: {total_products - products_multiple_prices:,} ({(total_products - products_multiple_prices)/total_products*100:.1f}%)")

# Examples of products with good price variation
print("\n Examples of products with multiple price points:")
multi_price_products = product_price_counts[product_price_counts > 5].head(5)

for product_id in multi_price_products.index:
    product_data = product_price_quantity[product_price_quantity['product_id'] == product_id]
    n_prices = len(product_data)
    price_range = f"{product_data['price'].min():.2f} - {product_data['price'].max():.2f}"
    total_qty = product_data['quantity'].sum()
    bucket = product_data['bucket'].iloc[0]
    print(f"{bucket:<20} {n_prices} prices, {total_qty:>4} units sold, range: {price_range}")


In [ ]:
# Checking for multi seller products - if this exists then we have IVs for BLP estimation

print("Multi seller product check for BLP instruments")

# Counting sellers per product
product_seller_counts = orders_full_extended.groupby('product_id')['seller_id'].nunique()

multi_seller = (product_seller_counts > 1).sum()
single_seller = (product_seller_counts == 1).sum()
total_products = len(product_seller_counts)

print("\n Seller competition")
print(f"Total products: {total_products:,}")
print(f"Products sold by multiple sellers: {multi_seller:,} ({multi_seller/total_products*100:.1f}%)")
print(f"Products sold by single seller: {single_seller:,} ({single_seller/total_products*100:.1f}%)")

# Detailed breakdown
seller_dist = product_seller_counts.value_counts().sort_index()
print("\n Seller distribution")
for n_sellers, count in seller_dist.head(20).items():
    print(f"{n_sellers} sellers: {count:,} products")

# Checking for price variation across sellers for same product
print("\n Price variation across sellers for same product")

multi_seller_products = product_seller_counts[product_seller_counts > 1].index

if len(multi_seller_products) > 0:
    # sample 5 multi-seller products to check price variation
    sample_products = np.random.choice(multi_seller_products, size=min(5, len(multi_seller_products)), replace=False)

    for product_id in sample_products:
        product_data = orders_full_extended[orders_full_extended['product_id'] == product_id]
        seller_prices = product_data.groupby('seller_id')['price'].mean()

        if len(seller_prices) > 1:
            cv = seller_prices.std() / seller_prices.mean() if seller_prices.mean() > 0 else 0
            print(f"Product: {len(seller_prices)} sellers, prices {seller_prices.min():.2f} - {seller_prices.max():.2f}, CV={cv:.2f}")

# BLP estimation feasibility

multi_seller_pct = multi_seller/total_products * 100

if multi_seller_pct >=  20:
    print(f"\n Strong Result: {multi_seller_pct:.2f}% multi_seller products.")
    print("BLP estimation is feasible with these products as potential instruments.")
elif multi_seller_pct >= 10:
    print(f"\n Moderate Result: {multi_seller_pct:.2f}% multi_seller products.")
    print("BLP estimation may be possible but with limited instruments. May need additional instrumental variables or focus on simpler models.")
else:
    print(f"\n Weak Result: {multi_seller_pct:.2f}% multi_seller products.")
    print("BLP estimation may not be feasible due to insufficient multi-seller products for instruments. Consider alternative approaches for elasticity estimation.")

In [ ]:
# Bucket level demand exploration

print("\n Bucket level demand exploration: looking for clear demand patterns across buckets")

# Loading and inspecting the data
print("Dataset Overview:")
print(f"Total observations: {len(bucket_demand_monthly):,}")
print(f"Unique buckets: {bucket_demand_monthly['bucket'].nunique()}")
print(f"Time periods: {bucket_demand_monthly['year_month'].nunique()}")
print(f"Date range: {bucket_demand_monthly['year_month'].min()} to {bucket_demand_monthly['year_month'].max()}")

print("\n Data structure:")
print(bucket_demand_monthly.head(10))

print("\n Summary statistics by bucket:")
bucket_summary = bucket_demand_monthly.groupby('bucket').agg({
    'quantity': ['count', 'mean', 'std', 'min', 'max'],
    'price': ['mean', 'std', 'min', 'max']
}).round(2)
print(bucket_summary)

# Visualize demand patterns
print("\n Creating visualizations of demand patterns across buckets")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Bucket Level Demand Analysis for Elasticity Estimation', fontsize=16, y = 1.00)

# Focusing on top 4 buckets from our validation analysis above
top_buckets = ['LEISURE_LIFESTYLE', 'HOME_ESSENTIALS', 'PERSONAL_CARE', 'ELECTRONICS_TECH'] 
colors = sns.color_palette("husl", len(top_buckets))

# 1. Quantity distribution by bucket
ax1 = axes[0, 0]
for i, bucket in enumerate(top_buckets):
    data = bucket_demand_monthly[bucket_demand_monthly['bucket'] == bucket].sort_values('year_month')
    ax1.plot(range(len(data)), data['quantity'], marker='o', label=bucket, color=colors[i], linewidth=2)
ax1.set_xlabel('Month Index', fontsize=11)
ax1.set_ylabel('Quantity (Units Sold)', fontsize=11)
ax1.set_title('Quantity Trends Over Time', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# 2. Price distribution by bucket
ax2 = axes[0, 1]
for i, bucket in enumerate(top_buckets):
    data = bucket_demand_monthly[bucket_demand_monthly['bucket'] == bucket].sort_values('year_month')
    ax2.plot(range(len(data)), data['price'], marker='o', label=bucket, color=colors[i], linewidth=2)
ax2.set_xlabel('Month Index', fontsize=11)
ax2.set_ylabel('Average Price (BRL)', fontsize=11)
ax2.set_title('Price Trends Over Time', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Revenue trends by bucket
ax3 = axes[0, 2]
bucket_demand_monthly['revenue'] = bucket_demand_monthly['quantity'] * bucket_demand_monthly['price']
for i, bucket in enumerate(top_buckets):
    data = bucket_demand_monthly[bucket_demand_monthly['bucket'] == bucket].sort_values('year_month')
    ax3.plot(range(len(data)), data['revenue'], marker='o', label=bucket, color=colors[i], linewidth=2)
ax3.set_xlabel('Month Index', fontsize=11)
ax3.set_ylabel('Revenue (BRL)', fontsize=11)
ax3.set_title('Revenue Trends Over Time', fontsize=12, fontweight='bold')
ax3.legend(fontsize=9)

# Quantity vs Price scatter plot
ax4 = axes[1, 0]
for i, bucket in enumerate(top_buckets):
    data = bucket_demand_monthly[bucket_demand_monthly['bucket'] == bucket]
    ax4.scatter(data['price'], data['quantity'], label=bucket, color=colors[i], alpha=0.6, s = 100)
ax4.set_xlabel('Price (BRL)', fontsize=11)
ax4.set_ylabel('Quantity (Units Sold)', fontsize=11)
ax4.set_title('Quantity vs Price by Bucket', fontsize=12, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

# log-log plot for elasticity estimation
ax5 = axes[1, 1]
for i, bucket in enumerate(top_buckets):
    data = bucket_demand_monthly[bucket_demand_monthly['bucket'] == bucket]
    # Removing any zeros or negatives before log transformation
    data_clean = data[(data['price'] > 0) & (data['quantity'] > 0)]
    ax5.scatter(np.log(data['price']), np.log(data['quantity']), label=bucket, color=colors[i], alpha=0.6, s = 100)
ax5.set_xlabel('Log(Price)', fontsize=11)
ax5.set_ylabel('Log(Quantity)', fontsize=11)
ax5.set_title('Log-Log Plot for Elasticity Estimation', fontsize=12, fontweight='bold')
ax5.legend(fontsize=9)
ax5.grid(True, alpha=0.3)
ax5.annotate('Slope = Price Elasticity', xy=(00.5, 0.95), xycoords='axes fraction', 
             fontsize=10, ha='left', va='top', bbox = dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Distribution of quantity for top buckets
ax6 = axes[1, 2]
for i, bucket in enumerate(top_buckets):
    data = bucket_demand_monthly[bucket_demand_monthly['bucket'] == bucket]
    ax6.hist(data['quantity'], bins=20, alpha=0.6, label=bucket, color=colors[i])
ax6.set_xlabel('Quantity (Units Sold)', fontsize=11)
ax6.set_ylabel('Frequency', fontsize=11)
ax6.set_title('Distribution of Quantity Sold by Bucket', fontsize=12, fontweight='bold')
ax6.legend(fontsize=9)
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Check data quality for elasticity estimation
print("\n Data quality check for elasticity estimation: looking for sufficient variation and reasonable patterns in quantity and price")

print("\n checking for zeros and missing values in price and quantity")
print(f"Price - zeros: {(bucket_demand_monthly['price'] == 0).sum()}, missing: {bucket_demand_monthly['price'].isna().sum()}")
print(f"Quantity - zeros: {(bucket_demand_monthly['quantity'] == 0).sum()}, missing: {bucket_demand_monthly['quantity'].isna().sum()}")

print("\n Price variation check: looking for reasonable variation in price across buckets")
for bucket in top_buckets:
    bucket_data = bucket_demand_monthly[bucket_demand_monthly['bucket'] == bucket]
    price_range = bucket_data['price'].max() - bucket_data['price'].min()
    price_cv = bucket_data['price'].std() / bucket_data['price'].mean() if bucket_data['price'].mean() > 0 else 0
    print(f"{bucket:<25} Price range: {price_range:.2f}, CV: {price_cv:.2f} {'Good' if price_cv > 0.1 else 'Low'}")

print("\n Time series completeness check: looking for consistent monthly data across buckets")
for bucket in top_buckets:
    bucket_data = bucket_demand_monthly[bucket_demand_monthly['bucket'] == bucket]
    expected_months = bucket_demand_monthly['year_month'].nunique()
    actual_months = bucket_data['year_month'].nunique()
    print(f"{bucket:<25} Months with data: {actual_months}/{expected_months} {'Complete' if actual_months >= expected_months * 0.9 else 'Incomplete'}")

#Data ready for elasticity estimation with these buckets
#Next steps
#Log-transform quantity and price
#Run log-log regression: log(Q) = alpha + beta * log(P) + controls
#beta coefficient will give price elasticity of demand for each bucket
#Add time fixed effects and bucket fixed effects to control for seasonality and bucket-specific factors


# Key observations:
# Quantity Trends: 
# Clear upward trends overtime suggesting growing market. 
# Home_essentials has a strady growth
# Leisure_lifestyle has more volatility but strong growth
# All buckets show reasonable variation
# Price Trends:
# Prices are relatively stable with some fluctuations.
# Leisure_lifestyle ~ 145 BRL on average
# Home_essentials ~ 100 BRL on average
# Electronics has some spikes but mostly stable
# Revenue Trends:
# Revenue is growing across all buckets, with Leisure_lifestyle leading in revenue generation.
# Log-Log Plot:
# Points clustered by bucket suggest distinct demand patterns.
# Data Quality Checks:
# No zeros or missing values in price and quantity, which is good for log-log regression.
# Price variation is sufficient across buckets, with CVs above 0.1, indicating good conditions for elasticity estimation.
# 21 - 23 months of data per bucket, which is sufficient for time series analysis and controlling for seasonality in regression models.

In [ ]:
# Bucket level elasticity estimation

import statsmodels.api as sm
import statsmodels.formula.api as smf

print("\n Bucket level elasticity estimation: running log-log regression to estimate price elasticity of demand for each bucket")

# Preparing regression data

# Create log-transformed variables for price and quantity
bucket_demand_monthly['log_price'] = np.log(bucket_demand_monthly['price'])
bucket_demand_monthly['log_quantity'] = np.log(bucket_demand_monthly['quantity'])

# Create time index for fixed effects
bucket_demand_monthly['month_index'] = pd.Categorical(bucket_demand_monthly['year_month']).codes

# Filter to viable buckets for elasticity estimation (excluding small_appliances and misc)
viable_buckets = ['LEISURE_LIFESTYLE', 'HOME_ESSENTIALS', 'PERSONAL_CARE', 'ELECTRONICS_TECH',
                  'AUTO_TOOLS', 'OFFICE_STATIONARY', 'FASHIOIN_APPAREL', 'FOOD_BEVERAGE']

regression_data = bucket_demand_monthly[bucket_demand_monthly['bucket'].isin(viable_buckets)].copy()

print(f"Total observations: {len(regression_data)}")
print(f"Buckets in regression data: {regression_data['bucket'].nunique()}")
print(f"Time periods in rgression data: {regression_data['year_month'].nunique()}")

# Regression 1: Pooled OLS (Baseline)

print("\nMODEL 1: Pooled OLS (No Fixed Effects)")

model1 = smf.ols('log_quantity ~ log_price', data=regression_data).fit()

print("\n Results - Model1")
print(f"Elasticity (beta): {model1.params['log_price']:.3f}")
print(f"Std Error: {model1.bse['log_price']:.3f}")
print(f"t-statistic: {model1.tvalues['log_price']:.3f}")
print(f"p-value: {model1.pvalues['log_price']:.4f}")
print(f"R-squared: {model1.rsquared:.3f}")
print(f"Observations: {model1.nobs:.0f}")

# Regression 2: Time Fixed Effects
print("\nModel 2: With time fixed effects")

model2 = smf.ols('log_quantity ~ log_price + C(month_index)', data=regression_data).fit()

print("\n Results - Model2")
print(f"Elasticity (beta): {model2.params['log_price']:.3f}")
print(f"Std Error: {model2.bse['log_price']:.3f}")
print(f"t-statistic: {model2.tvalues['log_price']:.3f}")
print(f"p-value: {model2.pvalues['log_price']:.4f}")
print(f"R-squared: {model2.rsquared:.3f}")
print(f"Observations: {model2.nobs:.0f}")

# Regression 3: Bucket Fixed Effects
print("\nModel 3: With Bucket Fixed Effects")

model3 = smf.ols('log_quantity ~ log_price + C(bucket)', data=regression_data).fit()

print("\n Results - Model3")
print(f"Elasticity (beta): {model3.params['log_price']:.3f}")
print(f"Std Error: {model3.bse['log_price']:.3f}")
print(f"t-statistic: {model3.tvalues['log_price']:.3f}")
print(f"p-value: {model3.pvalues['log_price']:.4f}")
print(f"R-squared: {model3.rsquared:.3f}")
print(f"Observations: {model3.nobs:.0f}")

# Regression 4: Full specification (Bucket + Time FE) 
print("\nModel 4: Full specification (Bucket + Time FE)")

model4 = smf.ols('log_quantity ~ log_price + C(bucket) + C(month_index)', data=regression_data).fit()

print("\n Results - Model4")
print(f"Elasticity (beta): {model4.params['log_price']:.3f}")
print(f"Std Error: {model4.bse['log_price']:.3f}")
print(f"t-statistic: {model4.tvalues['log_price']:.3f}")
print(f"p-value: {model4.pvalues['log_price']:.4f}")
print(f"R-squared: {model4.rsquared:.3f}")
print(f"Observations: {model4.nobs:.0f}")

# 95% Confidence Interval
conf_int = model4.conf_int(alpha=0.05)
ci_lower = conf_int.loc['log_price', 0]
ci_upper = conf_int.loc['log_price', 1]
print(f"95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]")

# Comparison Table

print("\nModel Comparison")

comparison = pd.DataFrame({
    'Model': ['Pooled_OLS', 'Time  FE', 'Bucket FE', 'Bucket + Time FE'],
    'Elasticity': [
        model1.params['log_price'],
        model2.params['log_price'],
        model3.params['log_price'],
        model4.params['log_price']
    ],
    'Std Error':[
        model1.bse['log_price'],
        model2.bse['log_price'],
        model3.bse['log_price'],
        model4.bse['log_price']
    ],
    'R-squared':[
        model1.rsquared,
        model2.rsquared,
        model3.rsquared,
        model4.rsquared
    ]
})

print(comparison.round(3).to_string(index=False))

# Key Insights:
# Results showed positive coefficients in naive specifications (Models 1-3), reflecting market growth trends rather than price sensitivity. 
#After controlling for bucket and time fixed effects (Model 4), the estimated 
#elasticity was 0.09 (SE=0.19, p=0.65), statistically indistinguishable from 
#zero [95% CI: -0.29, 0.46].
# The positive coefficients state that the consumers are bnuying more as the price increases. This is counter intiuitive anf doesn't make economic sense for normal goods.
# This is coming mostly for aggregation. As individual bucket has many categories. For instance customers switching between 
# high/low price products within bucket (e.g., toys vs watches within LEISURE_LIFESTYLE). So we are moving to category-level elasticity estimation.

In [ ]:
# Category-level elasticity estimation
print("Creating category level demand dataset")

# Aggregating the category month level data

category_demand_monthly = orders_full_extended.groupby(
    ['product_category_name_english', 'year_month', 'bucket']).agg({
        'order_id': 'count', # Quantity
        'price': 'mean', # Average price
        'freight_value': 'mean',
        'review_score': 'mean',
        'product_id': 'nunique', # Product diversity
        'seller_id': 'nunique', # Seller competition
        'customer_unique_id': 'nunique' # Customer reach
    }).reset_index()

# Renaming the quantity for clarity
category_demand_monthly.rename(columns={'order_id': 'quantity'}, inplace=True)

print(f"Total number of observations: {len(category_demand_monthly):,}")
print(f"Unique categories: {category_demand_monthly['product_category_name_english'].nunique()}")
print(f"Time periods: {category_demand_monthly['year_month'].nunique()}")
print(f"Buckets represented: {category_demand_monthly['bucket'].nunique()}")

# Create log-transformed variables
category_demand_monthly['log_quantity'] = np.log(category_demand_monthly['quantity'])
category_demand_monthly['log_price'] = np.log(category_demand_monthly['price'])

# Creating time series index 
category_demand_monthly['month_index'] = pd.Categorical(category_demand_monthly['year_month']).codes

# Filter to categories with sufficient data
category_counts = category_demand_monthly.groupby('product_category_name_english').size()

# Keeping categories with at least 12 months of data
sufficient_categories = category_counts[category_counts>=12].index

category_regression_data = category_demand_monthly[
    category_demand_monthly['product_category_name_english'].isin(sufficient_categories)].copy()

print(f" Categories with 12+ observations: {len(sufficient_categories)}")
print(f"Total observations for regression: {len(category_regression_data):,}")


# Summary by bucket
bucket_summary = category_regression_data.groupby('bucket').agg({
    'product_category_name_english': 'nunique',
    'quantity': ['count', 'mean'],
    'price': 'mean'
}).round(2)
print(bucket_summary)

# Focusing on top 5 buckets in terms of observations and product categories
focus_buckets = ['LEISURE_LIFESTYLE', 'HOME_ESSENTIALS', 'ELECTRONICS_TECH', 'AUTO_TOOLS', 'FASHION_APPAREL']

category_focus = category_regression_data[category_regression_data['bucket'].isin(focus_buckets)].copy()

print(category_focus.columns)

print("\n Focus dataset (top 5 buckets)")
print(f"Observations: {len(category_focus):,}")
print(f"Categories: {category_focus['product_category_name_english'].nunique()}")
print(f"Buckets: {category_focus['bucket'].nunique()}")

# Distribution by bucket
print(category_focus.groupby('bucket').size().sort_values(ascending=False))

# Top categories
top_cats = category_focus.groupby('product_category_name_english')['quantity'].sum().sort_values(ascending=False).head(15)
print(top_cats)

print(f"Avg prices: {category_focus.groupby('product_category_name_english')['price'].mean()}")

# Saving the data into pickle
category_focus.to_pickle('../data/processed/category_demand_monthly_focus.pkl')


In [ ]:
# Category level elasticity estimation with the selected data

print("Category level price elasticity estimation")

# Pooled estimation (all categories)

print("POOLED MODEL: All categories with bucket fixed effects + time fixed effects")

pooled_model = smf.ols(
    'log_quantity ~ log_price + C(bucket) + C(month_index)', data = category_focus
).fit()

print("Pooled Elasticity")
print(f"Coefficient (beta): {pooled_model.params['log_price']:.3f}")
print(f"Std error: {pooled_model.bse['log_price']:.3f}")
print(f"t-statistic: {pooled_model.tvalues['log_price']:.3f}")
print(f"p-value: {pooled_model.pvalues['log_price']:.4f}")
print(f"R-squared: {pooled_model.rsquared:.3f}")
print(f"Observations: {pooled_model.nobs:.0f}")

# 95% CI 
conf_int = pooled_model.conf_int(alpha = 0.5)
ci_lower = conf_int.loc['log_price', 0]
ci_upper = conf_int.loc['log_price', 1]
print(f"95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]")

if pooled_model.pvalues['log_price'] < 0.05:
    print(f"\n Statistically significant at 5% level!")
    if pooled_model.params['log_price'] < 0:
        print(f"NEGATIVE elasticity (as expected!) = {pooled_model.params['log_price']:.3f}")
    else:
        print(f"Positive elasticity (unexpected) = {pooled_model.params['log_price']:.3f}")
else:
    print(f"\nNot statistically significant")

# Category specific elasticities
print(" Category-Specific elasticities")

category_results = []

# Get top 20 categories by observation count
top_20_categories = category_focus.groupby('product_category_name_english').size().sort_values(ascending=False).head(20).index

for category in top_20_categories:
    cat_data = category_focus[category_focus['product_category_name_english'] == category]
    
    # Need at least 15 observations for reliable estimation
    if len(cat_data) < 15:
        continue
    
    try:
        # Estimate with time FE (bucket is constant within category)
        model = smf.ols('log_quantity ~ log_price + C(month_index)', data=cat_data).fit()
        
        elasticity = model.params['log_price']
        std_err = model.bse['log_price']
        t_stat = model.tvalues['log_price']
        p_value = model.pvalues['log_price']
        
        # 95% CI
        conf_int = model.conf_int(alpha=0.05)
        ci_lower = conf_int.loc['log_price', 0]
        ci_upper = conf_int.loc['log_price', 1]
        
        # Get bucket
        bucket = cat_data['bucket'].iloc[0]
        
        # Store results
        category_results.append({
            'Category': category,
            'Bucket': bucket,
            'Elasticity': elasticity,
            'Std_Error': std_err,
            't_statistic': t_stat,
            'p_value': p_value,
            'CI_Lower': ci_lower,
            'CI_Upper': ci_upper,
            'Significant': p_value < 0.05,
            'R_squared': model.rsquared,
            'Observations': model.nobs
        })
        
    except Exception as e:
        print(f"  ⚠️  {category}: Error - {str(e)[:50]}")
        continue

# ----------------------------------------
# Create results dataframe
# ----------------------------------------
elasticity_df = pd.DataFrame(category_results)

if len(elasticity_df) > 0:
    # Sort by elasticity (most elastic first)
    elasticity_df = elasticity_df.sort_values('Elasticity')
    
    print(f"\n Successfully estimated {len(elasticity_df)} category elasticities")
    print("\n" + "-"*100)
    print(f"{'Category':<30} {'Bucket':<20} {'Elasticity':>10} {'Std Err':>10} {'p-value':>10} {'Sig':>5}")
    print("-"*100)
    
    for _, row in elasticity_df.iterrows():
        sig_marker = '***' if row['p_value'] < 0.01 else '**' if row['p_value'] < 0.05 else '*' if row['p_value'] < 0.10 else ''
        print(f"{row['Category']:<30} {row['Bucket']:<20} {row['Elasticity']:>10.3f} {row['Std_Error']:>10.3f} {row['p_value']:>10.4f} {sig_marker:>5}")
    
    # ----------------------------------------
    # Summary statistics
    # ----------------------------------------
    print("\n" + "="*60)
    print("Summary Statistics")
    print("="*60)
    
    print(f"\nElasticity Distribution:")
    print(f"Mean: {elasticity_df['Elasticity'].mean():.3f}")
    print(f"Median: {elasticity_df['Elasticity'].median():.3f}")
    print(f"Std Dev: {elasticity_df['Elasticity'].std():.3f}")
    print(f"Min: {elasticity_df['Elasticity'].min():.3f} ({elasticity_df.loc[elasticity_df['Elasticity'].idxmin(), 'Category']})")
    print(f"Max: {elasticity_df['Elasticity'].max():.3f} ({elasticity_df.loc[elasticity_df['Elasticity'].idxmax(), 'Category']})")
    
    print(f"\nStatistical Significance:")
    print(f"Significant at 1%: {(elasticity_df['p_value'] < 0.01).sum()}")
    print(f"Significant at 5%: {(elasticity_df['p_value'] < 0.05).sum()}")
    print(f"Significant at 10%: {(elasticity_df['p_value'] < 0.10).sum()}")
    print(f"Not significant: {(elasticity_df['p_value'] >= 0.10).sum()}")
    
    # Check for negative elasticities
    negative_elasticities = (elasticity_df['Elasticity'] < 0).sum()
    print(f"\nElasticity Signs:")
    print(f"Negative (expected): {negative_elasticities} ({negative_elasticities/len(elasticity_df)*100:.1f}%)")
    print(f"Positive (unexpected): {len(elasticity_df) - negative_elasticities}")
    
    # ----------------------------------------
    # By bucket
    # ----------------------------------------
    print("\n Elasticity by Bucket:")
    bucket_stats = elasticity_df.groupby('Bucket').agg({
        'Elasticity': ['count', 'mean', 'std', 'min', 'max']
    }).round(3)
    print(bucket_stats)
    
    # ----------------------------------------
    # Save results
    # ----------------------------------------
    elasticity_df.to_csv('../outputs/category_elasticities.csv', index=False)
else:
    print("\n No categories had sufficient data for estimation")


# Simpler model - just price, no time FE
simple_results = []

for category in top_20_categories:
    cat_data = category_focus[category_focus['product_category_name_english'] == category]
    
    if len(cat_data) >= 15:
        try:
            # Simple regression: log(Q) ~ log(P) only
            model = smf.ols('log_quantity ~ log_price', data=cat_data).fit()
            
            simple_results.append({
                'Category': category,
                'Elasticity': model.params['log_price'],
                'Std_Error': model.bse['log_price'],
                'p_value': model.pvalues['log_price']
            })
        except:
            continue

simple_df = pd.DataFrame(simple_results)
print("\nSIMPLE MODEL (no time FE):")
print(simple_df.sort_values('Elasticity'))

simple_df_sorted = simple_df.sort_values('Elasticity')
simple_df_sorted.to_csv('../outputs/category_elasticities_simple.csv', index=False)

print("\n Simple model results saved!")
print("\Summary:")
print(f"Negative elasticities: {(simple_df['Elasticity'] < 0).sum()} of {len(simple_df)}")
print(f"Significant (p<0.05): {(simple_df['p_value'] < 0.05).sum()}")
print(f"Mean elasticity: {simple_df['Elasticity'].mean():.2f}")
print(f"Median elasticity: {simple_df['Elasticity'].median():.2f}")

In [ ]:
# ============================================
# CATEGORY ELASTICITY WITH CONTROLS
# ============================================

print("\n" + "="*60)
print("CATEGORY ELASTICITY WITH FREIGHT & REVIEW CONTROLS")
print("="*60)

# Create log freight (handle zeros)
category_focus['log_freight'] = np.log(category_focus['freight_value'] + 1)  # +1 to avoid log(0)

# ----------------------------------------
# POOLED MODEL WITH CONTROLS
# ----------------------------------------
print("\nPOOLED MODEL: Price + Freight + Reviews")
print("-"*60)

pooled_controls = smf.ols(
    'log_quantity ~ log_price + log_freight + review_score',
    data=category_focus
).fit()

print(f"\nResults:")
print(f"Price elasticity: {pooled_controls.params['log_price']:.3f} (p={pooled_controls.pvalues['log_price']:.4f})")
print(f"Freight elasticity: {pooled_controls.params['log_freight']:.3f} (p={pooled_controls.pvalues['log_freight']:.4f})")
print(f"Review effect: {pooled_controls.params['review_score']:.3f} (p={pooled_controls.pvalues['review_score']:.4f})")
print(f"R-squared: {pooled_controls.rsquared:.3f}")

# ----------------------------------------
# CATEGORY-SPECIFIC WITH CONTROLS
# ----------------------------------------
print("\n" + "="*60)
print("CATEGORY-SPECIFIC ELASTICITIES (WITH CONTROLS)")
print("="*60)

controlled_results = []

# Get top 20 categories
top_20_categories = category_focus.groupby('product_category_name_english')['quantity'].sum().sort_values(ascending=False).head(20).index

for category in top_20_categories:
    cat_data = category_focus[category_focus['product_category_name_english'] == category]
    
    if len(cat_data) >= 15:
        try:
            # Model with controls
            model = smf.ols(
                'log_quantity ~ log_price + log_freight + review_score',
                data=cat_data
            ).fit()
            
            bucket = cat_data['bucket'].iloc[0]
            
            controlled_results.append({
                'Category': category,
                'Bucket': bucket,
                'Price_Elasticity': model.params['log_price'],
                'Price_SE': model.bse['log_price'],
                'Price_pval': model.pvalues['log_price'],
                'Freight_Elasticity': model.params['log_freight'],
                'Review_Effect': model.params['review_score'],
                'R_squared': model.rsquared,
                'Observations': int(model.nobs)
            })
        except:
            continue

# Create results dataframe
controlled_df = pd.DataFrame(controlled_results)

if len(controlled_df) > 0:
    # Sort by price elasticity
    controlled_df = controlled_df.sort_values('Price_Elasticity')
    
    print(f"\n Estimated {len(controlled_df)} categories\n")
    
    # Display key columns
    display_cols = ['Category', 'Bucket', 'Price_Elasticity', 'Price_pval', 
                    'Freight_Elasticity', 'Review_Effect', 'R_squared']
    print(controlled_df[display_cols].to_string(index=False))
    
    # Summary statistics
    print("\n" + "="*60)
    print("Summary Statistics")
    print("="*60)
    
    print(f"\n Price Elasticity:")
    print(f"Mean: {controlled_df['Price_Elasticity'].mean():.3f}")
    print(f"Median: {controlled_df['Price_Elasticity'].median():.3f}")
    print(f"Negative: {(controlled_df['Price_Elasticity'] < 0).sum()} of {len(controlled_df)} ({(controlled_df['Price_Elasticity'] < 0).sum()/len(controlled_df)*100:.1f}%)")
    print(f"Significant (p<0.05): {(controlled_df['Price_pval'] < 0.05).sum()}")
    
    print(f"\n Freight Elasticity:")
    print(f"Mean: {controlled_df['Freight_Elasticity'].mean():.3f}")
    print(f"Negative: {(controlled_df['Freight_Elasticity'] < 0).sum()} of {len(controlled_df)}")
    
    print(f"\n Review Score Effect:")
    print(f"Mean: {controlled_df['Review_Effect'].mean():.3f}")
    print(f"Positive: {(controlled_df['Review_Effect'] > 0).sum()} of {len(controlled_df)}")
    
    print(f"\n Model Fit:")
    print(f"Average R-squared: {controlled_df['R_squared'].mean():.3f}")
    print(f"Min R-squared: {controlled_df['R_squared'].min():.3f}")
    print(f"Max R-squared: {controlled_df['R_squared'].max():.3f}")
    
    # Compare to simple model
    print("\n" + "="*60)
    print("COMPARISON: SIMPLE vs WITH CONTROLS")
    print("="*60)
    
    # Merge with simple results
    comparison = simple_df.merge(
        controlled_df[['Category', 'Price_Elasticity', 'R_squared']], 
        on='Category', 
        suffixes=('_simple', '_controlled')
    )

    # Fix column names after merge
    comparison.rename(columns={
        'Elasticity': 'Elasticity_simple',
        'Price_Elasticity': 'Price_Elasticity_controlled',
        'R_squared': 'R_squared_controlled'
    }, inplace=True)
    
    print(f"\nHow controls changed elasticity estimates:")
    print(f"Mean change: {(comparison['Price_Elasticity_controlled'] - comparison['Elasticity_simple']).mean():.3f}")
    print(f"Mean R-squared improvement: {(comparison['R_squared_controlled'] - comparison['Elasticity_simple'].apply(lambda x: 0)).mean():.3f}")
    
    print("\nTop 5 categories with biggest changes:")
    comparison['Change'] = comparison['Price_Elasticity_controlled'] - comparison['Elasticity_simple']
    print(comparison.nlargest(5, 'Change')[['Category', 'Elasticity_simple', 'Price_Elasticity_controlled', 'Change']])
    
    # Save results
    controlled_df.to_csv('../outputs/category_elasticities_controlled.csv', index=False)

In [ ]:
## ============================================
# FINAL ELASTICITY SUMMARY
# ============================================

print("\n" + "="*80)
print("FINAL ELASTICITY SUMMARY: SIMPLE vs CONTROLLED")
print("="*80)

# Merge the two result sets
final_summary = simple_df.merge(
    controlled_df[['Category', 'Bucket', 'Price_Elasticity', 'Price_pval', 'R_squared']], 
    on='Category',
    how='inner',  # Only keep categories in BOTH datasets
    suffixes=('_simple', '_controlled')
)

# Rename columns for clarity
final_summary.rename(columns={
    'Elasticity': 'Elasticity_Simple',
    'p_value': 'Pval_Simple',
    'Bucket_simple': 'Bucket',
    'Price_Elasticity': 'Elasticity_Controlled',
    'Price_pval': 'Pval_Controlled',
    'R_squared': 'R_squared_Controlled'
}, inplace=True)

# Keep only relevant columns
final_summary = final_summary[[
    'Category', 'Bucket', 
    'Elasticity_Simple', 'Pval_Simple',
    'Elasticity_Controlled', 'Pval_Controlled', 
    'R_squared_Controlled'
]]

# Add significance markers
final_summary['Sig_Simple'] = final_summary['Pval_Simple'].apply(
    lambda p: '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else ''))
)
final_summary['Sig_Controlled'] = final_summary['Pval_Controlled'].apply(
    lambda p: '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else ''))
)

# Calculate change
final_summary['Change'] = final_summary['Elasticity_Controlled'] - final_summary['Elasticity_Simple']

# Identify ROBUST findings (negative & significant in BOTH)
final_summary['Robust'] = (
    (final_summary['Elasticity_Simple'] < 0) & 
    (final_summary['Elasticity_Controlled'] < 0) &
    (final_summary['Pval_Simple'] < 0.05) &
    (final_summary['Pval_Controlled'] < 0.05)
)

# Sort by controlled elasticity
final_summary = final_summary.sort_values('Elasticity_Controlled')

print(f"\nCategories in both models: {len(final_summary)}\n")

# Display main results
display_cols = ['Category', 'Bucket', 'Elasticity_Simple', 'Sig_Simple', 
                'Elasticity_Controlled', 'Sig_Controlled', 'Change', 'Robust']
print(final_summary[display_cols].to_string(index=False))

# ============================================
# ROBUST FINDINGS
# ============================================
print("\n" + "="*80)
print("ROBUST FINDINGS (Negative & Significant in both Models)")
print("="*80)

robust = final_summary[final_summary['Robust']]

if len(robust) > 0:
    print(f"\n {len(robust)} categories show robust negative elasticities:\n")
    print(robust[['Category', 'Bucket', 'Elasticity_Simple', 'Elasticity_Controlled', 
                  'R_squared_Controlled']].to_string(index=False))
    
    print("\n These categories are:")
    print("Negative in both specifications")
    print("Statistically significant (p<0.05) in BOTH")
    print("High Confidence elasticity estimates")
else:
    print("\n No categories meet all robust criteria")
    print("But this is informative - shows data limitations")

# ============================================
# SUMMARY STATISTICS
# ============================================
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

print("\nSimple Model:")
print(f"Negative elasticities: {(final_summary['Elasticity_Simple'] < 0).sum()} of {len(final_summary)} ({(final_summary['Elasticity_Simple'] < 0).sum()/len(final_summary)*100:.1f}%)")
print(f"Significant (p<0.05): {(final_summary['Pval_Simple'] < 0.05).sum()}")
print(f"Mean elasticity: {final_summary['Elasticity_Simple'].mean():.3f}")

print("\n Controlled Model:")
print(f"Negative elasticities: {(final_summary['Elasticity_Controlled'] < 0).sum()} of {len(final_summary)} ({(final_summary['Elasticity_Controlled'] < 0).sum()/len(final_summary)*100:.1f}%)")
print(f"Significant (p<0.05): {(final_summary['Pval_Controlled'] < 0.05).sum()}")
print(f"Mean elasticity: {final_summary['Elasticity_Controlled'].mean():.3f}")
print(f"Mean R-squared: {final_summary['R_squared_Controlled'].mean():.3f}")

print("\n Agreement between models:")
both_negative = ((final_summary['Elasticity_Simple'] < 0) & (final_summary['Elasticity_Controlled'] < 0)).sum()
print(f"Both negative: {both_negative} of {len(final_summary)} ({both_negative/len(final_summary)*100:.1f}%)")

print("\nLargest changes (controls matter most):")
print(final_summary.nlargest(5, 'Change')[['Category', 'Elasticity_Simple', 'Elasticity_Controlled', 'Change']].to_string(index=False))

# ============================================
# SAVE RESULTS
# ============================================
final_summary.to_csv('../outputs/elasticity_final_summary.csv', index=False)

In [ ]:
# NOTE ON ANALYTICAL APPROACH:
# Initial analysis explored nested logit estimation for bucket-level demand modeling.
# Cells 13-16 document this exploration including the bucket transition matrix analysis.
# Finding: With only 3.12% repeat purchase rate and 53.1% of repeat customers switching
# buckets between purchases, the switching behavior is insufficient to support a
# well-identified nested logit structure.
# Decision: Propensity score modeling (cells 17-28) is used instead to estimate
# repeat purchase probability, which forms the basis of the CLV calculation.
# The transition matrix results are retained here for transparency and as supporting
# evidence for the sequential purchasing framework developed in notebook 05.

# Nested logit data preparation

print("Nested logit data preparation")

print("Creating bucket choice structure")

nested_data = orders_full_extended[
    ['order_id', 'customer_unique_id', 'product_id', 'bucket', 
     'price', 'freight_value', 'review_score', 'order_purchase_timestamp']
    ]

# Removing any missing values
nested_data = nested_data.dropna(subset=['bucket','price'])
print(f"Total purchases: {len(nested_data)}")
print(f"Unique customers: {nested_data['customer_unique_id'].nunique():,}")
print(f"Unique products: {nested_data['product_id'].nunique():,}")
print(f"Buckets: {nested_data['bucket'].nunique():,}")

# Bucket level statistics

# Average price per bucket
bucket_avg_price = nested_data.groupby('bucket')['price'].mean().to_dict()
nested_data['bucket_avg_price'] = nested_data['bucket'].map(bucket_avg_price)

# Number of products per bucket
bucket_num_products = nested_data.groupby('bucket')['product_id'].nunique().to_dict()
nested_data['bucket_num_products'] = nested_data['bucket'].map(bucket_num_products)

# Average review score per bucket
bucket_avg_review = nested_data.groupby('bucket')['review_score'].mean().to_dict()
nested_data['bucket_avg_review'] = nested_data['bucket'].map(bucket_avg_review)

# Focusing on top 5 buckets with most amount of data
focus_buckets = ['LEISURE_LIFESTYLE', 'HOME_ESSENTIALS', 'ELECTRONICS_TECH',
                 'AUTO_TOOLS', 'FASHION_APPAREL']

nested_focus = nested_data[nested_data['bucket'].isin(focus_buckets)].copy()

print("\n Focus dataset")
print(f"Number of purchases: {len(nested_focus)}")
print(f"Customers: {nested_focus['customer_unique_id'].nunique():,}")
print(f"Buckets: {nested_focus['bucket'].nunique()}")


# Bucket distribution
print("\n Purchase distribution by bucket:")
print(nested_focus['bucket'].value_counts().sort_values(ascending=False))

# Saving prepared data
nested_focus.to_pickle('../data/processed/nested_logit_data.pkl')

In [ ]:
# Bucket transition matrix

print("Bucket transition analysis")

# Identifying repeat customers within the nested_focus dataset

customer_purchase_counts = nested_focus.groupby('customer_unique_id').size()
repeat_customers = customer_purchase_counts[customer_purchase_counts >= 2].index

repeat_purchases = nested_focus['customer_unique_id'].isin(repeat_customers).copy()
repeat_purchases = repeat_purchases.reset_index(drop=True)  # Fix indexing

print(f"Total customers: {nested_focus['customer_unique_id'].nunique():,}")
print(f"Repeat customers: {len(repeat_customers):,}")
print(f"Repeat rate: {len(repeat_customers)/nested_focus['customer_unique_id'].nunique()*100:,}")
print(f"Repeat purchases: {len(repeat_purchases):,}")


# Calculate bucket transition
print("\n calculating bucket-to-bucket transition")

transitions_list=[]

for customer_id in repeat_customers:
    # Get all purchases for this customer, sorted by time
    customer_orders = nested_focus[nested_focus['customer_unique_id'] == customer_id].sort_values('order_purchase_timestamp')
    
    # Get bucket sequence
    bucket_sequence = customer_orders['bucket'].values
    
    # Create transitions
    for i in range(len(bucket_sequence) - 1):
        transitions_list.append({
            'from_bucket': bucket_sequence[i],
            'to_bucket': bucket_sequence[i+1],
            'same_bucket': bucket_sequence[i] == bucket_sequence[i+1]
        })

transition_df = pd.DataFrame(transitions_list)

print("\n Transition patterns:")
print(f"Total transactions: {len(transition_df):,}")
print(f"Same bucket (loyalty): {transition_df['same_bucket'].sum():,} ({transition_df['same_bucket'].mean()*100:.1f})")
print(f"Different bucket (switching):{(~transition_df['same_bucket']).sum():,} ({(~transition_df['same_bucket']).mean()*100:.1f}%)")


# Creating transition matrix
print("Bucket transition matrix")

# Count transitions
transition_matrix = pd.crosstab(
    transition_df['from_bucket'],
    transition_df['to_bucket'],
    normalize='index' # Row percentages
)

print("\n probability of transitioning from bucket (row) to bucket (column)")
print(transition_matrix.round(3))

# Insights
print("Key substitution insights")

# Diagonal (loyalty rates)
print("\n Loyalty rates (staying in the same bucket)")
for bucket in transition_matrix.index:
    if bucket in transition_matrix.columns:
        loyalty = transition_matrix.loc[bucket, bucket]
        print(f" {bucket:<25} {loyalty*100:>5.1f}%")

# Top cross bucket switches
print("\n Top cross-bucket switches:")
cross_bucket = transition_df[~transition_df['same_bucket']]
top_switches = cross_bucket.groupby(['from_bucket', 'to_bucket']).size().sort_values(ascending=False).head(10)

for (from_b, to_b), count in top_switches.items():
    pct = count / len(cross_bucket) * 100
    prob = count / len(transition_df[transition_df['from_bucket'] == from_b]) * 100
    print(f"  {from_b:<20} to {to_b:<20} {count:>4} switches ({prob:>4.1f}% of {from_b} customers)")

# Which buckets are closest substitutes?
print("\n Substitution Pairs (excluding loyalty):")
# Get off-diagonal (cross-bucket) probabilities
substitution_pairs = []
for from_b in transition_matrix.index:
    for to_b in transition_matrix.columns:
        if from_b != to_b:  # Exclude diagonal
            prob = transition_matrix.loc[from_b, to_b]
            substitution_pairs.append({
                'From': from_b,
                'To': to_b,
                'Probability': prob
            })

substitution_df = pd.DataFrame(substitution_pairs).sort_values('Probability', ascending=False).head(10)
print(substitution_df.to_string(index=False))


# Saving the results
transition_matrix.to_csv('../outputs/bucket_transition_matrix.csv')
transition_df.to_csv('../outputs/bucket_transitions.csv', index=False)

# Key Insights
# Overall Patterns:
# 93% loyalty rate - customers mostly stay in same bucket
# Only 7% Switch buckets
# Home_Essentials: 95.6%, Electronics_Tech: 92.5%, Leisure_Lifestyle: 89.8%, Auto_Tools: 89.0%, Fashion_Apparel: 81.7%
# TOP SUBSTITUTION PATTERNS:
#1. HOME_ESSENTIALS and LEISURE_LIFESTYLE (Bidirectional)
# HOME to LEISURE: 148 switches (2.1%)
# LEISURE to HOME: 139 switches (5.0%)
# These are substitutes. People switch between home and leisure purchases

# 2. HOME_ESSENTIALS and ELECTRONICS_TECH
# ELECTRONICS to HOME: 78 switches (3.4%)
# HOME to ELECTRONICS: 72 switches (1.0%)
# Complementary purchases - buying tech for home

# 3. LEISURE_LIFESTYLE and ELECTRONICS_TECH
# LEISURE to ELECTRONICS: 71 switches (2.5%)
# ELECTRONICS to LEISURE: 66 switches (2.9%)
# Gaming connection? (consoles counted as ELECTRONICS, games as LEISURE)

# Business Insights
# Strongest Cross-Bucket Substitutes:
# FASHION_APPAREL to LEISURE_LIFESTYLE (8%) - Fashion shoppers explore leisure
# FASHION_APPAREL to HOME_ESSENTIALS (7%) - Fashion to home goods
# AUTO_TOOLS to HOME_ESSENTIALS (6%) - Auto buyers also buy home goods

# Implications:
# If you raise HOME_ESSENTIALS prices, expect some switching to LEISURE (2.1%)
# If you raise FASHION prices, expect 8% to switch to LEISURE
# But overall switching is low (7%) - bucket loyalty is strong

# Pricing implications:
# Limited Cross-Bucket Substitution:

# 93% stay in same bucket to Each bucket operates somewhat independently
# Pricing in one bucket has minimal impact on others
# Within-bucket competition matters more than cross-bucket

# Exception - Fashion is Different:
# Only 82% loyalty (lowest)
# 18% explore other buckets
# More price-sensitive, more exploratory


In [ ]:
# Visualize Transition Matrix

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("\n Creating transition matrix visualization")

fig, ax = plt.subplots(figsize=(10, 8))

# Create heatmap
sns.heatmap(
    transition_matrix, 
    annot=True,           # Show numbers
    fmt='.2f',            # 2 decimal places
    cmap='YlOrRd',        # Yellow to Red color scheme
    cbar_kws={'label': 'Transition Probability'},
    linewidths=0.5,
    linecolor='gray',
    ax=ax,
    vmin=0,
    vmax=1
)

ax.set_title('Bucket-to-Bucket Transition Matrix\n(Probability customer switches from row bucket to column bucket)', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('To Bucket (Next Purchase)', fontsize=12, fontweight='bold')
ax.set_ylabel('From Bucket (Previous Purchase)', fontsize=12, fontweight='bold')

# Rotate labels for readability
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('../outputs/images/bucket_transition_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


# Create loyalty vs switching chart

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Loyalty rates
loyalty_rates = []
for bucket in transition_matrix.index:
    if bucket in transition_matrix.columns:
        loyalty_rates.append({
            'Bucket': bucket,
            'Loyalty': transition_matrix.loc[bucket, bucket] * 100
        })

loyalty_df = pd.DataFrame(loyalty_rates).sort_values('Loyalty', ascending=True)

ax1.barh(range(len(loyalty_df)), loyalty_df['Loyalty'], color='steelblue', edgecolor='black')
ax1.set_yticks(range(len(loyalty_df)))
ax1.set_yticklabels(loyalty_df['Bucket'])
ax1.set_xlabel('Loyalty Rate (%)', fontsize=11, fontweight='bold')
ax1.set_title('Bucket Loyalty Rates\n(% staying in same bucket)', fontsize=12, fontweight='bold')
ax1.axvline(x=93, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Overall Average (93%)')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, v in enumerate(loyalty_df['Loyalty']):
    ax1.text(v + 1, i, f'{v:.1f}%', va='center', fontweight='bold')

# Chart 2: Top substitution flows
top_10_subs = substitution_df.head(10).copy()
top_10_subs['Label'] = top_10_subs['From'].str[:8] + ' to ' + top_10_subs['To'].str[:8]
top_10_subs = top_10_subs.sort_values('Probability')

ax2.barh(range(len(top_10_subs)), top_10_subs['Probability'] * 100, color='coral', edgecolor='black')
ax2.set_yticks(range(len(top_10_subs)))
ax2.set_yticklabels(top_10_subs['Label'], fontsize=9)
ax2.set_xlabel('Switching Probability (%)', fontsize=11, fontweight='bold')
ax2.set_title('Top 10 Cross-Bucket Switches\n(excluding same-bucket)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, v in enumerate(top_10_subs['Probability'] * 100):
    ax2.text(v + 0.2, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/images/bucket_loyalty_and_switching.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# CATEGORY-LEVEL TRANSITIONS (WITHIN BUCKETS)
# ============================================

print("\n" + "="*60)
print("CATEGORY TRANSITIONS WITHIN BUCKETS")
print("="*60)

# ----------------------------------------
# First, add category names to nested_focus
# ----------------------------------------
print("\n Preparing data with category names")

# Merge category names from orders_full_extended
nested_with_category = nested_focus.merge(
    orders_full_extended[['order_id', 'product_category_name_english']],
    on='order_id',
    how='left'
)

# Remove any missing categories
nested_with_category = nested_with_category.dropna(subset=['product_category_name_english'])

print(f"Purchases with category info: {len(nested_with_category):,}")
print(f"Unique categories: {nested_with_category['product_category_name_english'].nunique()}")

# ----------------------------------------
# Focus on top 3 buckets with most categories
# ----------------------------------------
focus_buckets_nested = ['HOME_ESSENTIALS', 'LEISURE_LIFESTYLE', 'ELECTRONICS_TECH', 'AUTO_TOOLS', 'FASHION_APPAREL']

print(f"\n Analyzing category switching WITHIN buckets...")
print(f"Focus buckets: {', '.join(focus_buckets_nested)}")

# Get repeat customers who stayed in same bucket
repeat_in_bucket = []

for customer_id in repeat_customers:
    customer_orders = nested_with_category[
        nested_with_category['customer_unique_id'] == customer_id
    ].sort_values('order_purchase_timestamp')
    
    # Get bucket and category sequences
    buckets = customer_orders['bucket'].values
    categories = customer_orders['product_category_name_english'].values
    
    # Check if customer has any consecutive purchases in same bucket
    for i in range(len(buckets) - 1):
        if buckets[i] == buckets[i+1] and buckets[i] in focus_buckets_nested:
            # Same bucket - record category transition
            repeat_in_bucket.append({
                'bucket': buckets[i],
                'from_category': categories[i],
                'to_category': categories[i+1],
                'same_category': categories[i] == categories[i+1]
            })

within_bucket_df = pd.DataFrame(repeat_in_bucket)

print(f"\n Within-bucket transitions found: {len(within_bucket_df):,}")

if len(within_bucket_df) > 0:
    # Overall within-bucket patterns
    print(f"\n Overall within-bucket patterns:")
    print(f"Same category (loyalty): {within_bucket_df['same_category'].sum():,} ({within_bucket_df['same_category'].mean()*100:.1f}%)")
    print(f"Different category (switching): {(~within_bucket_df['same_category']).sum():,} ({(~within_bucket_df['same_category']).mean()*100:.1f}%)")
    
    # ----------------------------------------
    # Analyze each bucket separately
    # ----------------------------------------
    for bucket in focus_buckets_nested:
        bucket_transitions = within_bucket_df[within_bucket_df['bucket'] == bucket]
        
        if len(bucket_transitions) < 10:
            print(f"\n {bucket}: Insufficient data ({len(bucket_transitions)} transitions)")
            continue
        
        print("\n" + "="*60)
        print(f"{bucket}")
        print("="*60)
        
        print(f"\n Total transitions: {len(bucket_transitions):,}")
        print(f"Same category: {bucket_transitions['same_category'].sum():,} ({bucket_transitions['same_category'].mean()*100:.1f}%)")
        print(f"Different category: {(~bucket_transitions['same_category']).sum():,} ({(~bucket_transitions['same_category']).mean()*100:.1f}%)")
        
        # Create transition matrix for this bucket
        if len(bucket_transitions) >= 20:  # Need enough data
            category_matrix = pd.crosstab(
                bucket_transitions['from_category'],
                bucket_transitions['to_category'],
                normalize='index'
            )
            
            # Show top categories only (5x5 matrix for readability)
            top_from = bucket_transitions['from_category'].value_counts().head(5).index
            top_to = bucket_transitions['to_category'].value_counts().head(5).index
            
            # Filter matrix to top categories
            category_matrix_top = category_matrix.loc[
                category_matrix.index.isin(top_from),
                category_matrix.columns.isin(top_to)
            ]
            
            if len(category_matrix_top) > 0:
                print(f"\n Category Transition Matrix (Top 5x5):")
                print(category_matrix_top.round(3))
            
            # Category loyalty rates
            print(f"\n Category Loyalty Rates (within {bucket}):")
            loyalty_rates = []
            for cat in category_matrix.index:
                if cat in category_matrix.columns:
                    loyalty = category_matrix.loc[cat, cat]
                    count = len(bucket_transitions[bucket_transitions['from_category'] == cat])
                    if count >= 3:  # Only show if enough data
                        loyalty_rates.append({
                            'Category': cat,
                            'Loyalty': loyalty,
                            'Count': count
                        })
            
            if loyalty_rates:
                loyalty_df = pd.DataFrame(loyalty_rates).sort_values('Loyalty', ascending=False)
                print("\nTop 10 categories by loyalty:")
                for _, row in loyalty_df.head(10).iterrows():
                    print(f"  • {row['Category']:<35} {row['Loyalty']*100:>5.1f}% (n={row['Count']:.0f})")
            
            # Top category switches
            print(f"\n Top 10 Category Switches (within {bucket}):")
            cross_cat = bucket_transitions[~bucket_transitions['same_category']]
            if len(cross_cat) > 0:
                top_switches = cross_cat.groupby(['from_category', 'to_category']).size().sort_values(ascending=False).head(10)
                for (from_c, to_c), count in top_switches.items():
                    print(f"  {from_c:<30} → {to_c:<30} {count:>3} switches")
        
        # Save bucket-specific results
        bucket_transitions.to_csv(f'../outputs/category_transitions_{bucket.lower()}.csv', index=False)
    
    # ----------------------------------------
    # Save overall results
    # ----------------------------------------
    within_bucket_df.to_csv('../outputs/category_transitions_within_buckets.csv', index=False)
    print("\n Results saved to: ../outputs/category_transitions_within_buckets.csv")

else:
    print("\n No within-bucket transitions found (customers don't repeat within same bucket)")

# Key Insights
# Overall Pattern: 
# 48,687 within-bucket transitions (customers buying again in same bucket)
# 94.9% stay in same category within bucket!
# Only 5.1% switch categories within bucket
# CATEGORY LOYALTY BY BUCKET:
# ELECTRONICS_TECH: 97.4% (Highest category loyalty)
# computers_accessories: 98.7%
# telephony: 98.2%
# consoles_games: 98.0%
# People know what tech they need!

# FASHION_APPAREL: 96.9%
# fashion_bags: 98.5%
# male_clothing: 100%
# shoes: 100%
# Fashion shoppers stick to their category!

#AUTO_TOOLS: 96.2%
# signaling_security: 99.6%
# construction_tools: 98.3%
# auto: 97.8%
# Car owners buy car stuff!

# LEISURE_LIFESTYLE: 95.9%
# pet_shop: 98.8%
# sports_leisure: 98.2%
# watches_gifts: 97.9%
# Pet owners keep buying pet stuff!

# HOME_ESSENTIALS: 93.5% (Lowest, but still high!)
# housewares: 96.1%
# garden_tools: 95.5%
# furniture_decor: 94.0%
# bed_bath_table: 93.7%
# Most switching happens here (6.5%)


# TOP SUBSTITUTION PATTERNS (The 5.1% who switch):
# Within HOME_ESSENTIALS:
# furniture_decor and bed_bath_table (bidirectional, ~220 switches each way)
# bed_bath_table and housewares (~75 switches each way)
# furniture_decor and garden_tools (~82 switches each way)
# Pattern: Home improvement shoppers rotate between furniture, bedding, and housewares

# Within LEISURE_LIFESTYLE:
# sports_leisure to watches_gifts (20 switches)
# cool_stuff and toys (bidirectional)
# toys and sports_leisure (small)
#Pattern: Minimal switching - pet owners stay with pets, sports stay with sports

# Within ELECTRONICS_TECH:
# electronics and computers_accessories (~22 switches each way)
# telephony and computers_accessories (~16 switches each way)
# Pattern: Tech ecosystem purchases (phone to computer accessories)

# BUSINESS IMPLICATIONS:
# 1. HYPER-SEGMENTED CUSTOMER BASE:
# Customers are:
# 93% loyal to BUCKET
# 95% loyal to CATEGORY within bucket
# = 88% buy exact same category again

# This means:
# Extremely narrow purchase patterns
# High predictability
# Target marketing is highly effective

# 2. PRICING STRATEGY:
# Since category loyalty is 95%:
# Limited within-bucket price competition
# Can price categories independently (even within same bucket)
# Elasticity matters more than substitution

# 3. CROSS-SELL IS HARD:
# Only 5.1% switch categories within bucket:
# Cross-selling furniture to bed_bath customers? Hard (6% switch)
# Cross-selling toys to pet owners? Very hard (1% switch)
# Better strategy: Retention >> Cross-sell

# 4. HOME_ESSENTIALS = EXCEPTION:
# 6.5% category switching (highest):
# furniture_decor and bed_bath_table and housewares form a "home improvement cluster"
# This is your one cross-sell opportunity
# Bundle these 3 categories together!

In [ ]:
# Propensity score modeling. Here we analyse what predicts whether a customer will make a repeat purchase.
# This is important because we know there is only 3.12% repeat rate and there is a massive opportunity if we can increase it.
# Understanding the repeat drivers helps us focus on retention efforts
# CLV calculation requires repeat probability.

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.model_selection import cross_validate, cross_val_predict

# Load customer level data
# We need customer id, first purchase, repeat indicator

# Loading and creating the dataset from orders
orders = pd.read_csv('../data/olist_orders_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
products_english = pd.read_csv('../data/product_category_name_translation.csv')

products = products.merge(products_english, on = 'product_category_name', how='inner')

print(products.head())

print("Orders shape:", orders.shape)
print("Customers shape:", customers.shape)
print("\nCustomers columns:")
print(customers.columns.tolist())

# Check the mapping
print("\nCustomer ID mapping check:")
print(f"Unique customer_id in orders: {orders['customer_id'].nunique()}")
print(f"Unique customer_unique_id in customers: {customers['customer_unique_id'].nunique()}")

# Map customer_id to customer_unique_id
orders_with_unique = orders.merge(
    customers[['customer_id', 'customer_unique_id']], 
    on='customer_id',
    how='left'
)

print(f"\nAfter merge - orders with customer_unique_id: {orders_with_unique['customer_unique_id'].notna().sum()}")


In [ ]:
# Calculating the repeat rate

# Merge to get full transaction data with true customer ID
transactions = order_items.merge(
    orders_with_unique[['order_id', 'customer_id', 'customer_unique_id', 
                         'order_purchase_timestamp', 'order_status']],
    on='order_id'
)

transactions = transactions.merge(
    products[['product_id', 'product_category_name_english']],
    on='product_id',
    how='left'
)

# Convert timestamp
transactions['order_date'] = pd.to_datetime(transactions['order_purchase_timestamp'])

# Filter delivered orders only
transactions = transactions[transactions['order_status'] == 'delivered'].copy()

print(f"\nTotal transactions (delivered): {len(transactions)}")
print(f"Unique customer_unique_id: {transactions['customer_unique_id'].nunique()}")


# Counting orders per true customer
customer_order_counts = transactions.groupby('customer_unique_id')['order_id'].nunique().reset_index()
customer_order_counts.rename(columns={'order_id': 'num_orders'}, inplace=True)

print("\nOrder count distribution:")
print(customer_order_counts['num_orders'].value_counts().sort_index())

# Calculating the true repeat rate
customer_order_counts['is_repeat'] = (customer_order_counts['num_orders'] > 1).astype(int)

true_repeat_rate = customer_order_counts['is_repeat'].mean()
print(f"\n True repeat rate: {true_repeat_rate:.2%}")

print("\n Repeat customer breakdown")
print(customer_order_counts['is_repeat'].value_counts())
print("\n Percentages")
print(customer_order_counts['is_repeat'].value_counts(normalize=True))

In [ ]:
# Dropping duplicates
customers_unique = customers.drop_duplicates('customer_unique_id', keep='first')
print(f"Unique customers: {len(customers_unique)}")

# Creating features from the first purchase
delivered_transactions = transactions[transactions['order_status'] == 'delivered'].copy()
print(f"Delivered transactions: {len(delivered_transactions)}")

# Getting first order for each true customer
first_orders = delivered_transactions.sort_values('order_date').groupby('customer_unique_id', as_index=False).first()

print(f"\n First orders identified: {len(first_orders)}")
print(f"Unique customers: {first_orders['customer_unique_id'].nunique()}")

# Verify no duplicates
assert len(first_orders) == first_orders['customer_unique_id'].nunique(), "Duplicates found!"
print(" No duplicates!")

# Get the order_id of each customer's first order
first_order_ids = first_orders[['customer_unique_id', 'order_id']]

# Get all items from those first orders and sum them
first_order_items = delivered_transactions.merge(
    first_order_ids,
    on=['customer_unique_id', 'order_id'],
    how='inner'
)

print(f"\n First order items: {len(first_order_items)}")

# Aggregate to customer level
first_order_values = first_order_items.groupby('customer_unique_id', as_index=False).agg({
        'price': 'sum', # Total order value
        'freight_value': 'mean', # Avg freight
        'product_id': 'count', # Number of items
        'order_id': 'first', # Keeping order_id for review merging
        'order_date': 'first',
        'product_category_name_english': 'first'
    })

first_order_values.rename(columns={'product_id': 'num_items'}, inplace=True)
print(f"Aggregated: {len(first_order_values)}, unique: {first_order_values['customer_unique_id'].nunique()}")

print("\n First order summary statistics")
print(first_order_values[['price', 'freight_value', 'num_items']].describe())

print(f"\nFirst order values shape: {first_order_values.shape}")
print(f"Unique customers: {first_order_values['customer_unique_id'].nunique()}")

# Verify no duplicates after aggregation
assert len(first_order_values) == first_order_values['customer_unique_id'].nunique(), \
    "Duplicates after aggregation"
print(" No duplicates after aggregation")

# Getting review scores
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
print(f"Total reviews: {len(reviews)}")
print(f"unique order_id: {reviews['order_id'].nunique()}")

# Found some duplicate reviews
dup_reviews = reviews[reviews.duplicated('order_id', keep=False)].sort_values('order_id')
print(f"\nExample duplicate order IDs (first 10):")
print(dup_reviews.head(10)[['order_id', 'review_score', 'review_comment_title']])

# Check if duplicate reviews have different scores
score_variance = reviews.groupby('order_id')['review_score'].nunique()
orders_with_diff_scores = (score_variance > 1).sum()
print(f"\nOrders with different review scores: {orders_with_diff_scores}")

# Keeping first review per order
reviews = reviews.drop_duplicates('order_id', keep = 'first')

# Merging the review score with the first orders
first_order_values = first_order_values.merge(
    reviews[['order_id', 'review_score']],
    on = 'order_id',
    how = 'left'
)
print(f"After reviews: {len(first_order_values)}")
print(f"Reviews available: {first_order_values['review_score'].notna().sum()}/{len(first_order_values)}")

# Extract temporal features
first_order_values['day_of_week'] = first_order_values['order_date'].dt.dayofweek
first_order_values['month'] = first_order_values['order_date'].dt.month
first_order_values['year'] = first_order_values['order_date'].dt.year

# Add customer location
first_order_values = first_order_values.merge(
    customers_unique[['customer_unique_id', 'customer_state']],
    on='customer_unique_id',
    how='left'
)

print(f"After location: {len(first_order_values)}, unique: {first_order_values['customer_unique_id'].nunique()}")

# VERIFY NO DUPLICATES
assert len(first_order_values) == first_order_values['customer_unique_id'].nunique(), \
    f"Duplicates {len(first_order_values)} rows vs {first_order_values['customer_unique_id'].nunique()} unique"
print("No duplicates")

print("\nFeatures created:")
print(first_order_values.columns.tolist())
print(f"\nFinal shape: {first_order_values.shape}")
print(f"Unique customers: {first_order_values['customer_unique_id'].nunique()}")

In [ ]:
# Merging and preparing data
model_data = first_order_values.merge(
    customer_order_counts[['customer_unique_id', 'is_repeat', 'num_orders']],
    on = 'customer_unique_id',
    how = 'inner' # Only keep customers in both datasets
)

print(f"\nAfter filtering:")
print(f"model_data: {len(model_data)}")
print(f"Should match customer_order_counts: {len(customer_order_counts)}")

if len(model_data) == len(customer_order_counts):
    print("Perfect match!")
else:
    print(f"mismatch: {len(model_data)} vs {len(customer_order_counts)}")

print(f"Model Shape: {model_data.shape}")
print(f"Match check: {len(model_data)} == {len(first_order_values)} (should be equal length)")

print(f"\n Target distribution:")
print(model_data['is_repeat'].value_counts())
print("\n percentages")
print(model_data['is_repeat'].value_counts(normalize=True))

# Check for any missing values
print("Missing Values")
print(model_data.isnull().sum())

# Handeling missing values
median_review = model_data['review_score'].median()
model_data['review_score'] = model_data['review_score'].fillna(median_review)
model_data['product_category_name_english'] = model_data['product_category_name_english'].fillna('unknown')

print("\nAfter handling missing values:")
print(model_data.isnull().sum())

In [ ]:
# Feature Engineering

# Create categorical features
print("\n Top 10 product categories")
top_categories = model_data['product_category_name_english'].value_counts().head(10)
print(top_categories)

top_category_names = top_categories.index.to_list()
model_data['category_group'] = model_data['product_category_name_english'].apply(
    lambda x: x if x in top_category_names else 'other'
)

print("\nCategory grouping:")
print(model_data['category_group'].value_counts())

# One-hot encode categories
category_dummies = pd.get_dummies(model_data['category_group'], prefix='cat', drop_first=True)
print(f"\nCategory dummies: {category_dummies.shape[1]} features")

# Numeric features
numeric_features = model_data[[
    'price',              
    'freight_value',      
    'num_items',          
    'review_score',       
    'day_of_week',        
    'month'               
]].copy()

# Combine
X = pd.concat([numeric_features, category_dummies], axis=1)
y = model_data['is_repeat']

print(f"\n{'='*60}")
print(f"Feature Matrix: {X.shape}")
print(f"Features: {X.columns.tolist()}")
print(f"\nTarget: {y.sum():,} repeat ({y.mean():.2%}) | {(~y.astype(bool)).sum():,} one-time ({(1-y.mean()):.2%})")
print(f"{'='*60}")


In [ ]:
# Training the model

print("Training the logistic regression model")

print("Cross validation")

# Standardizing
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Train
model_cv = LogisticRegression(
    random_state=42,
    max_iter=1000,
    solver = 'lbfgs'
)

# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nRunning 5-fold cross-validation...")
print("(This evaluates on 20% held-out data in each fold)")

# Cross-validate with multiple metrics
cv_results = cross_validate(
    model_cv, 
    X_scaled, 
    y,
    cv=cv,
    scoring=['roc_auc', 'recall', 'precision'],
    n_jobs=-1,
    return_train_score=True
)

# ROC-AUC scores
print(f"\n ROC-AUC Scores (per fold):")
for fold, score in enumerate(cv_results['test_roc_auc'], 1):
    train_score = cv_results['train_roc_auc'][fold-1]
    print(f"  Fold {fold}: Test={score:.3f} | Train={train_score:.3f} | Diff={train_score-score:+.3f}")

print(f"\n Mean Test ROC-AUC: {cv_results['test_roc_auc'].mean():.3f} (±{cv_results['test_roc_auc'].std():.3f})")
print(f"Mean Train ROC-AUC: {cv_results['train_roc_auc'].mean():.3f}")
print(f"Overfitting: {cv_results['train_roc_auc'].mean() - cv_results['test_roc_auc'].mean():+.3f}")

if abs(cv_results['train_roc_auc'].mean() - cv_results['test_roc_auc'].mean()) < 0.02:
    print("Model generalizes well! (overfitting < 0.02)")
elif abs(cv_results['train_roc_auc'].mean() - cv_results['test_roc_auc'].mean()) < 0.05:
    print("Slight overfitting (0.02-0.05)")
else:
    print("Significant overfitting (>0.05)")

# Recall scores
print(f"\n Recall Scores:")
print(f"Mean Test Recall: {cv_results['test_recall'].mean():.3f} (±{cv_results['test_recall'].std():.3f})")

# Precision scores  
print(f"\n Precision Scores:")
print(f"Mean Test Precision: {cv_results['test_precision'].mean():.3f} (±{cv_results['test_precision'].std():.3f})")

# Store for documentation
cv_roc_auc_mean = cv_results['test_roc_auc'].mean()
cv_roc_auc_std = cv_results['test_roc_auc'].std()

print(f"\n{'='*60}")
print("Performance Estimation")
print(f"{'='*60}")
print(f"ROC-AUC: {cv_roc_auc_mean:.3f} ± {cv_roc_auc_std:.3f}")


# Predictions
y_pred_prob = cross_val_predict(
    model_cv,
    X,
    y,
    cv = cv,
    method='predict_proba'
)[:,1]

model_data['repeat_probability'] = y_pred_prob

print("Probability Distribution:")
print(f"Min: {model_data['repeat_probability'].min():.4f}")
print(f"Max: {model_data['repeat_probability'].max():.4f}")
print(f"Mean: {model_data['repeat_probability'].mean():.4f}")
print(f"Median: {model_data['repeat_probability'].median():.4f}")

# Verify these are honest predictions
print(f"\n{'='*60}")
print("Verification (Should match CV ROC-AUC):")
print(f"{'='*60}")
roc_auc_oof = roc_auc_score(y, y_pred_prob)
print(f"Out-of-fold ROC-AUC: {roc_auc_oof:.3f}")
print(f"Cross-val ROC-AUC: {cv_roc_auc_mean:.3f}")
print(f"Difference: {abs(roc_auc_oof - cv_roc_auc_mean):.3f}")

if abs(roc_auc_oof - cv_roc_auc_mean) < 0.001:
    print(" Perfect match. These are CV out-of-fold predictions.")

In [ ]:
# Feature importance
print("\nTraining full model for feature importance.")

scaler_full = StandardScaler()
X_scaled_full = scaler_full.fit_transform(X)

model_full = LogisticRegression(random_state=42, max_iter=1000, solver='lbfgs')
model_full.fit(X_scaled_full, y)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model_full.coef_[0],
    'abs_coef': np.abs(model_full.coef_[0]),
    'odds_ratio': np.exp(model_full.coef_[0])
}).sort_values('abs_coef', ascending=False)

print("\nTop 15 Features:")
print(feature_importance.head(15)[['feature', 'coefficient', 'odds_ratio']].to_string(index=False))
print(feature_importance[['feature', 'coefficient', 'odds_ratio']].to_string(index=False))
print(X.columns)


print("\n" + "="*60)
print("Top Drivers (increase repeat):")
print("="*60)
top_pos = feature_importance[feature_importance['coefficient'] > 0].head(5)
for _, row in top_pos.iterrows():
    print(f"  {row['feature']:30s} {row['coefficient']:+.3f} (OR={row['odds_ratio']:.2f})")

print("\n" + "="*60)
print("Top Barriers (decrease repeat):")
print("="*60)
top_neg = feature_importance[feature_importance['coefficient'] < 0].head(5)
for _, row in top_neg.iterrows():
    print(f"  {row['feature']:30s} {row['coefficient']:+.3f} (OR={row['odds_ratio']:.2f})")

In [ ]:
# Visualizations

fig, ax = plt.subplots(figsize=(10, 8))

top_15 = feature_importance.head(15).sort_values('coefficient')
colors = ['red' if x < 0 else 'green' for x in top_15['coefficient']]

ax.barh(range(len(top_15)), top_15['coefficient'], color=colors, alpha=0.7)
ax.set_yticks(range(len(top_15)))
ax.set_yticklabels(top_15['feature'])
ax.set_xlabel('Coefficient (Log Odds)', fontsize=12)
ax.set_title('Propensity to Repeat: Feature Importance\n(Green=Increases, Red=Decreases)', 
             fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/images/propensity_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# Add predictions to model_data
model_data['repeat_probability'] = y_pred_prob

In [ ]:
# Key Insights
# ROC-AUC is 0.583 this is modest but better than random 0.5
# Main challenge was the extreme imbalance of 97% one-time and 3% repeat

# Top drivers of repeat purchase:
# 1. Category bed_bath_table with odds ratio 1.27 meaning 27% higher odds of repeat. Makes sense as home essentials have on going needs
# 2. Furniture decor with odds ratio 1.22 meaning 22% higher odds of repeat purchase. Complete the room behavior
# 3. sports_leisure with odds ratio 1.20 meaning 20% higher odds of repeat purchase. Hobby enthusiasts return
# 4. num items with odds ratio 1.13. more items in the first order leads to 13% repeat purchase. Basket size matters.
# Top Barriers to repeat:
# 1. Price with odds ratio 0.92. Higher first purchase price meaning 8% lower odds of repeat purchase. Expensive first purchase less likely to return.
# 2. freight value with odds ratio 0.92. Higher shipping cost meaning 8% lower odds of repeat purchase. Shipping pain point.
# 3. Month. season effect.
# Business Insights:
# Encourage repeat purchases:
# 1. Focus on home essentials (bed/bath, furniture, housewares)
# 2. Encourage multi-item orders (bundling, complete the look addons)
# 3. Target hobby and sports enthuists (ongoing needs of equipemnt or gears)
# Reduce friction on the following items:
# 1. Lower prices on first purchase (customer acquisition stratgey)
# 2. Free or subsidizing shipping on the first order
# 3. Don't price out new customers with high entry point

In [ ]:
# Customer segmentation

print(model_data.columns)
print("Customer Segmentation")

print(f"Using out-of-fold predictions:")
print(f"Min: {model_data['repeat_probability'].min():.4f}")
print(f"Max: {model_data['repeat_probability'].max():.4f}")
print(f"Mean: {model_data['repeat_probability'].mean():.4f}")
print(f"Median: {model_data['repeat_probability'].median():.4f}")

model_data['propensity_rank'] = model_data['repeat_probability'].rank(
    ascending=False,
    method='first'
)

model_data['propensity_percentile'] = model_data['propensity_rank']/len(model_data)

# Segmenting: Top 20%, middle 30%, Bottom 50%
def segment_by_rank(percentile):
    if percentile <= 0.20:
        return 'High Propensity'
    elif percentile <= 0.50:
        return 'Medium Propensity'
    else:
        return 'Low Propensity'

model_data['segment'] = model_data['propensity_percentile'].apply(segment_by_rank)

print("Segment Sizes")
for segment in ['High Propensity', 'Medium Propensity', 'Low Propensity']:
    count = (model_data['segment']== segment).sum()
    pct = count / len(model_data)*100
    print(f"{segment:20s}: {count:6,} ({pct:5.1f}%)")


print("Segment Performance")
segment_results = []

for segment in ['High Propensity', 'Medium Propensity', 'Low Propensity']:
    seg_data = model_data[model_data['segment'] == segment]
    
    count = len(seg_data)
    pct_customers = count / len(model_data) * 100
    
    actual_repeats = int(seg_data['is_repeat'].sum())
    actual_rate = seg_data['is_repeat'].mean()
    pct_of_all_repeats = actual_repeats / 2801 * 100
    lift = actual_rate / 0.03
    
    avg_prob = seg_data['repeat_probability'].mean()
    min_prob = seg_data['repeat_probability'].min()
    max_prob = seg_data['repeat_probability'].max()
    
    avg_order = seg_data['price'].mean()
    avg_freight = seg_data['freight_value'].mean()
    avg_items = seg_data['num_items'].mean()
    avg_review = seg_data['review_score'].mean()
    
    print(f"\n{segment}:")
    print(f"Size:               {count:,} ({pct_customers:.1f}%)")
    print(f"Probability range:  {min_prob:.4f} - {max_prob:.4f} (avg: {avg_prob:.4f})")
    print(f"Actual repeat rate: {actual_rate:.2%}")
    print(f"Lift vs baseline:   {lift:.1f}x")
    print(f"Repeats captured:   {actual_repeats:,} ({pct_of_all_repeats:.1f}%)")
    print(f"Avg order value:   BRL {avg_order:.2f}")
    print(f"Avg freight:       BRL {avg_freight:.2f}")
    print(f"Avg items:          {avg_items:.2f}")
    print(f"Avg review score:   {avg_review:.2f}")
    
    # Store for later
    segment_results.append({
        'segment': segment,
        'count': count,
        'actual_rate': actual_rate,
        'repeats': actual_repeats,
        'avg_order': avg_order
    })

# Key insights
high_repeats = int(model_data[model_data['segment'] == 'High Propensity']['is_repeat'].sum())
high_med_repeats = int(model_data[model_data['segment'].isin(['High Propensity', 'Medium Propensity'])]['is_repeat'].sum())

print("Insights")
print(f"Top 20% captures:  {high_repeats:,} repeats ({high_repeats/2801*100:.1f}%)")
print(f"Top 50% captures:  {high_med_repeats:,} repeats ({high_med_repeats/2801*100:.1f}%)")
print(f"Bottom 50% has:    {2801-high_med_repeats:,} repeats ({(2801-high_med_repeats)/2801*100:.1f}%)")

efficiency = (high_repeats/2801) / 0.20
print(f"\n Targeting top 20% is {efficiency:.1f}x more efficient than random")

# Key findings:
# High Propensity top 20%: 4.40% repeat rate which is 1.5 times the baseline, Captures 29.3% of all repeat customers.
# Medium propensity 20-50% : 2.98% repeat rate, Captures 29.8% of repeats
# Low Propensity bottom 50% : 2.45% repeat rate, contains 40.8% of repeats but spread across 50% of customers

# Business Insights:
# High propensity customers: They order value is lower at first (105 BRL vs 178 BRL), 
#                            They order more items per order (1.39 vs 1.03), They seek lower shipping (16.84 BRL vs 23.86 BLR)
#                            They provided better reviews (4.28 vs 3.92)
#                            Clear pattern: multi-item, stisfied customers with lower-value orders
# Low Propensity customers: Higher first order value (178 BRL) expensive one-time purchases 
#                           Order fewer items (1.03) - mostly a single expensive item, Pay for higher shipping (23.86 BRL)
#                           Provided worse review. A clear pattern: expensive single item purchases with poor experience

In [ ]:
# Customer Lifetime Value Analysis
print("Customer Lifetime Value")

# Get average second purchase value for repeat customers
repeat_customers_clv = model_data[model_data['is_repeat'] == 1]['customer_unique_id']

# Finding the second purchases
second_purchases = (transactions[transactions['customer_unique_id'].isin(repeat_customers_clv)]
                   .sort_values('order_date')
                   .groupby('customer_unique_id')
                   .nth(1)  # Second purchase (Note: 0-indexed, so 1 = 2nd)
                   .reset_index())

avg_second_purchase = second_purchases['price'].sum() / len(second_purchases) if len(second_purchases) > 0 else 0

print(f"\nSecond purchase analysis:")
print(f"Customers who repeated: {len(repeat_customers):,}")
print(f"Avg second purchase value: BRL {avg_second_purchase:.2f}")
print(f"Avg first purchase value: BRL {model_data['price'].mean():.2f}")


# Calculate expected CLV by segment
print("Expected CLV by Segment:")

clv_results = []

for segment in ['High Propensity', 'Medium Propensity', 'Low Propensity']:
    seg_data = model_data[model_data['segment'] == segment]
    
    avg_first = seg_data['price'].mean()
    avg_prob = seg_data['repeat_probability'].mean()
    
    # Expected CLV = First Purchase + P(repeat) × Second Purchase
    expected_clv = avg_first + (avg_prob * avg_second_purchase)
    
    # Incremental value from repeat
    incremental = avg_prob * avg_second_purchase
    
    # Lifetime value premium vs Low segment
    low_clv = model_data[model_data['segment'] == 'Low Propensity']['price'].mean() + \
              (model_data[model_data['segment'] == 'Low Propensity']['repeat_probability'].mean() * avg_second_purchase)
    
    clv_premium = expected_clv - low_clv
    clv_premium_pct = (clv_premium / low_clv * 100) if low_clv > 0 else 0
    
    print(f"\n{segment}:")
    print(f" Avg first purchase: BRL{avg_first:.2f}")
    print(f" Avg repeat probability: {avg_prob:.2%}")
    print(f" Expected repeat value: BRL {incremental:.2f}")
    print(f" Expected CLV: BRL {expected_clv:.2f}")
    print(f" CLV premium vs Low: BRL {clv_premium:+.2f} ({clv_premium_pct:+.1f}%)")
    
    clv_results.append({
        'segment': segment,
        'expected_clv': expected_clv,
        'incremental': incremental
    })

# Total CLV across all customers
total_first_purchase = model_data['price'].sum()
total_expected_repeat = (model_data['repeat_probability'] * avg_second_purchase).sum()
total_clv = total_first_purchase + total_expected_repeat

print(f"\n{'='*60}")
print("TOTAL VALUE ANALYSIS:")
print(f"{'='*60}")
print(f"Total first purchase value: BRL {total_first_purchase:,.2f}")
print(f"Expected repeat purchase value: BRL {total_expected_repeat:,.2f}")
print(f"Total expected CLV: BRL {total_clv:,.2f}")
print(f"Repeat contribution to total value: {total_expected_repeat/total_clv*100:.1f}%")

# Value concentration
high_value = model_data[model_data['segment'] == 'High Propensity']['price'].sum() + \
             (model_data[model_data['segment'] == 'High Propensity']['repeat_probability'] * avg_second_purchase).sum()

print(f"\n{'='*60}")
print("VALUE CONCENTRATION:")
print(f"{'='*60}")
print(f"Top 20% of customers represent: BRL {high_value:,.2f} ({high_value/total_clv*100:.1f}% of total CLV)")

In [ ]:
#Strategic Interpretation:

# Current State:
# High Propensity: BRL 110 CLV, 4.5% repeat (EASIER to improve)
# Low Propensity:  BRL 181 CLV, 2.4% repeat (HARDER to improve)

# Retention Investment Recommendation:
# Invest: High & Medium propensity (50% of customers)
# Already engaged (4.5% & 3.0% repeat rates)
# Multi-item buyers (cross-sell opportunity)
# Happy customers (4.28 & 4.48 reviews)
# Lower friction (smaller orders, lower shipping)

# SKIP: Low propensity (50% of customers)
# One-time big purchases (BRL 178 avg)")
# Unhappy (3.92 reviews - investigate why)
# High friction (BRL 23.86 shipping)
# Hard to retain (2.4% baseline)

# Key Insight:")
# Lower CLV does not mean lower priority
# High propensity = easier retention = Better ROI"
# Focus budget on customers who are already engaged

print(f"\n Opportunity Sizing:")
high_customers = (model_data['segment'] == 'High Propensity').sum()
current_repeats = model_data[model_data['segment'] == 'High Propensity']['is_repeat'].sum()
current_rate = current_repeats / high_customers

# If we improve 4.5% to 6.0% (33% increase)
target_rate = 0.06
new_repeats = high_customers * target_rate
incremental_repeats = new_repeats - current_repeats

incremental_revenue = incremental_repeats * avg_second_purchase

print(f"  Current high-propensity repeats: {int(current_repeats):,} ({current_rate:.2%})")
print(f"  Target (33% improvement):        {int(new_repeats):,} (6.0%)")
print(f"  Incremental repeats:             {int(incremental_repeats):,}")
print(f"  Incremental revenue:             R$ {incremental_revenue:,.2f}")
print(f"  Worth spending ~R$ {incremental_revenue*0.5:,.2f} on retention!")


# Elasticity estimates from this notebook are used in notebook 05
# for profit-aware pricing optimization.
# Bundle viability is tested in notebook 04.
# Sequential purchase patterns are analyzed in notebook 05.